# Diabetic Retinopathy Detection with CNNs and Explainability

**Dataset:** APTOS 2019 Blindness Detection (Kaggle) — ~3,660 fundus images, 5 severity grades.

**Pipeline:** circle-crop + Ben Graham preprocessing → stratified 70/15/15 split → three-model comparison (ResNet-18 from scratch, EfficientNet-B0/B3 transfer learning) with class-weighted loss → QWK / per-class F1 / confusion matrices / referable-DR screening view → Grad-CAM explainability, including failure cases.

Runs top-to-bottom on a Colab **GPU runtime** (Runtime → Change runtime type → T4 GPU). Total runtime ≈ 1.5–2.5 h including download and preprocessing.

All library code lives in the `src/` package of the repository cloned below, so this notebook stays a readable experiment log.

## 1. Setup

In [ ]:
# Confirm GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
# Clone the project repository (private repo — Colab will prompt for a GitHub token if needed)
REPO_URL = 'https://github.com/OgaSir-Gianni/Diabetic-Retinopathy--Roehampton-Uni.git'

import os
if not os.path.exists('Diabetic-Retinopathy--Roehampton-Uni'):
    !git clone -q {REPO_URL}
%cd Diabetic-Retinopathy--Roehampton-Uni/dr-detection
!pip install -q -r requirements.txt

## 2. Data download

Requires a Kaggle API token (kaggle.com → Settings → Create New Token, a `KGAT_...` string). Paste it when prompted — it is not stored in the notebook.

The cell downloads the full-resolution public mirror of the APTOS 2019 data (`mariaherrerot/aptos2019`) and consolidates it into the official layout — same 3,662 labelled images, no competition-rules acceptance needed. If you have accepted the rules on the [competition page](https://www.kaggle.com/competitions/aptos2019-blindness-detection), you can drop the `--mirror` flag to download the official competition files instead; the pipeline is identical either way (the script verifies the label distribution matches the official data).

In [ ]:
import os
from getpass import getpass
if not os.environ.get('KAGGLE_API_TOKEN'):
    os.environ['KAGGLE_API_TOKEN'] = getpass('Kaggle API token (KGAT_...): ')

!python scripts/download_data.py --mirror

## 3. Preprocessing

Two cached variants are built once: `ben` (circle-crop + resize + Ben Graham Gaussian-blur subtraction) and `plain` (circle-crop + resize only). The `plain` cache is the control condition for the preprocessing ablation in Section 6.

In [ ]:
!python scripts/preprocess_data.py

## 4. Exploratory data analysis

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from src.config import CLASS_NAMES, RAW_DIR
from src.data import make_splits

df = make_splits()
counts = df['diagnosis'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([CLASS_NAMES[i] for i in counts.index], counts.values, color='tab:blue')
for i, v in enumerate(counts.values):
    ax.text(i, v + 20, f'{v}\n({v/len(df):.0%})', ha='center')
ax.set_ylabel('Images')
ax.set_title('APTOS 2019 class distribution — grade 0 dominates')
plt.show()

print('\nSplit sizes:')
print(pd.crosstab(df['diagnosis'], df['split'], margins=True))

In [ ]:
# Raw vs preprocessed: one example per grade (Figure for the report)
import cv2
from src.preprocessing import preprocess_image

fig, axes = plt.subplots(3, 5, figsize=(16, 10))
for k in range(5):
    id_code = df[df['diagnosis'] == k].iloc[0]['id_code']
    path = RAW_DIR / 'train_images' / f'{id_code}.png'
    raw = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    axes[0, k].imshow(raw);                                    axes[0, k].set_title(f'{CLASS_NAMES[k]}\nraw')
    axes[1, k].imshow(preprocess_image(path, apply_ben=False)); axes[1, k].set_title('crop + resize')
    axes[2, k].imshow(preprocess_image(path, apply_ben=True));  axes[2, k].set_title('+ Ben Graham')
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout(); plt.show()

## 5. Training — three-way model comparison

All runs share the same frozen split, class-weighted cross-entropy, AdamW + cosine schedule, and early stopping on validation QWK. Only the architecture/initialisation changes.

In [ ]:
# 5.1 Baseline: ResNet-18 trained from scratch (no transfer learning)
!python -m src.train --model resnet18_scratch --variant ben --num-workers 2

In [ ]:
# 5.2 Transfer learning: EfficientNet-B0 (ImageNet weights)
!python -m src.train --model efficientnet_b0 --variant ben --num-workers 2

In [ ]:
# 5.3 Transfer learning: EfficientNet-B3 (ImageNet weights, 300px, smaller batch)
!python -m src.train --model efficientnet_b3 --variant ben --batch-size 16 --num-workers 2

## 6. Ablation — does Ben Graham preprocessing help?

Identical training run to 5.2, but on the `plain` cache (no illumination normalisation).

In [ ]:
!python -m src.train --model efficientnet_b0 --variant plain --num-workers 2

## 7. Test-set evaluation

In [ ]:
RUNS = ['resnet18_scratch_ben', 'efficientnet_b0_ben',
        'efficientnet_b3_ben', 'efficientnet_b0_plain']

from src.evaluate import evaluate
all_metrics = {run: evaluate(run) for run in RUNS}

In [ ]:
# Headline comparison table (Table for the report)
import pandas as pd
rows = []
for run, m in all_metrics.items():
    rows.append({
        'run': run,
        'QWK': round(m['test_qwk'], 4),
        'accuracy': round(m['test_accuracy'], 4),
        'macro F1': round(m['macro_f1'], 4),
        'referable sens.': round(m['referable']['sensitivity'], 4),
        'referable spec.': round(m['referable']['specificity'], 4),
        'referable AUC': round(m['referable']['auc'], 4),
    })
results = pd.DataFrame(rows).set_index('run')
results

In [ ]:
# Learning curves and confusion matrices for each run
from IPython.display import Image as IPImage, display
from src.config import OUTPUT_DIR
for run in RUNS:
    display(IPImage(str(OUTPUT_DIR / run / 'curves.png')))
    display(IPImage(str(OUTPUT_DIR / run / 'confusion_matrix.png')))

## 8. Explainability — Grad-CAM

Panels for the best model: highest-confidence correct predictions per grade, and the most confident misclassifications. The question for the report: does the network attend to lesions (microaneurysms, haemorrhages, exudates) or to artefacts — and do confident errors show attention in the wrong place?

In [ ]:
BEST_RUN = results['QWK'].idxmax()
print('Best run by test QWK:', BEST_RUN)

from src.gradcam import generate_panels
generate_panels(BEST_RUN, per_class=2, n_errors=6)

In [ ]:
import glob
for path in sorted(glob.glob(str(OUTPUT_DIR / BEST_RUN / 'gradcam' / '*.png'))):
    display(IPImage(path))

## 9. Outputs

Everything the report cites is under `outputs/<run>/`: `history.json` (training log), `curves.png`, `metrics.json`, `confusion_matrix.png`, `predictions.csv`, and `gradcam/*.png`. Zip and download below to keep them after the Colab VM is recycled.

In [ ]:
!zip -qr outputs.zip outputs
from google.colab import files
files.download('outputs.zip')